In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import product
import scipy.stats as sc
import seaborn as sns
from math import floor

In [ ]:
class ECM:
  '''
  The implementation of ECM (Error Correction Model) estimation for panel data.
  ----
  ----
  PARAMETERS:
  ----
  - *df*: a Pandas DataFrame containing panel data. Make sure your data is structured in this exact order (by column index):\n
    0 - Spatial units. The data must be homogenous, e.g. only cities, contries, regions, etc.\n
    1 - Temporal units. The data must be homogenous, e.g. only years, months, quarters, etc. \n
    2 - Your target/endogenous variable. The data must not contain NaN values.\n
    3+ - Your exogenous variables. The data must not contain NaN values.\n
  - *effects*: Specify what effects your long-run model must have. The class will estimate both the short-run & long-run models.
    Enter one of the following keywords: "fix" | "rand". "rand" by default.
  - *trend*: Specify a trend of which order your target variable is. 0 by default - the data contains no trend.
  - *n_lags*: Specify the maximum amount of lags to test on. The estimator will choose the best lag amount based on AIC.
  - *method*: Specify the method of ECM estimation. \n
    Currently implemented methods:\n
    - MG (Mean Group) - Choose this method of you believe all your coefficients may be simply heterogenous.
    - CCEMG (Common Correlated Effects Mean Group) - Choose this one if you also believe that there is valid cross-sectional dependence in the data. \n
    Choose between the following keywords: ["MG", "CCEMG"]
  - *coint*: Specify which exogenous variables will be included in the long-run model (A.K.A. are conitegrated with the TARGET variable). 
      The rest will be included only in the short-run model.\n
      Enter: a string containing 1 single variable / a list of strings in the following format.\n
      (enumeration from left to right column-wise in your DataFrame):
      - coint = "x1"
      - coint = ["x1", "x3", ...]\n
      Defaults to "x1"
  ----
  RETURNS:
  --
  A python dictionary (dict) containing:
    - Long-run estimation results | key = "lr_res"
    - ECM (short-run) estimation results for each individual region | key = "sr_res"
    - A string representing the RE/FE-estimated long-run equation | key = "lr_eq"
    - A string representing the MG-estimated ECM (short-run) equation | key = "sr_eq"
  '''
  def __init__(self, df: pd.DataFrame, effects: str = 'rand', trend: int = 0, n_lags: int = 1, method: str = 'MG', coint: str | list[str] = 'x1') -> None:
    self.__df = df
    self.__eff = effects.lower()
    self.__t = trend
    self.__lag = n_lags
    self.__method = method.lower()
    self.__exog = len(df.columns[3:])
    self.__l =[]
    for i in range(1, self.__exog+1):
      self.__l.append(f'x{i}')
    self.__df.columns = ['SpUnit', 'time', 'target'] + self.__l
    if isinstance(coint, list):
      self.__lr_df = self.__df.copy(deep=True).loc[:, ['SpUnit', 'time', 'target', *coint]]
    elif isinstance(coint, str):
      self.__lr_df = self.__df.copy(deep=True).loc[:, ['SpUnit', 'time', 'target', coint]]
    else:
      raise TypeError('An invalid Type has been passed into the COINT parameter')
    if self.__t > 0:
      self.__lr_df = self.add_trend()
    self.__N = len(self.__df.SpUnit.unique())
    self.__T = len(self.__df.time.unique())
    if self.__lag > self.__T**(1/3):
      self.__lag = floor(self.__T**(1/3))
      if self.__lag < 1:
        self.__lag = 1
    self.__verify()
    self.__means = self.build_means()
    self.__lr = self.__estimate_lr()
    self.__sr = None
  
  def __verify(self) -> None:
    if self.__eff not in ['fix', 'rand']:
      raise ValueError('Non-Valid panel effects type entered!')
    if self.__trend < 0:
      raise ValueError('The Trend order cannot be lower than 0!')
    if self.__method not in ['mg', 'ccemg']:
      raise NotImplementedError('Either the estimation method has not been implemented yet, or it is invalid!')
    
  def add_trend(self) -> pd.DataFrame:
    lst = []
    for unit in self.__df.SpUnit.unique():
      subdf = self.__lr_df[self.__lr_df.SpUnit == unit]
      for i in range(1, self.__t+1):
          subdf[f't^{i}'] = np.linspace(1, len(self.__df.time.unique()), len(self.__df.time.unique()))**i
      lst.append(subdf)
    return pd.concat(lst)
        
  def build_means(self) -> pd.DataFrame:
    mn = self.__df.copy(deep=True)
    mn = mn.set_index('time')
    means = mn.groupby('time')[['target'] + self.__l].mean()
    return means
  
  def build_GLS(self, w_err: float) -> pd.DataFrame:
    re = self.__lr_df.copy(deep=True)
    sigma2 = np.sum((sm.OLS(re.iloc[:, 2], sm.add_constant(re.iloc[:, 3:])).fit()).resid**2) / (self.__N*self.__T - self.__exog)
    sigma_u = sigma2 - w_err
    if sigma_u <= 0:
      sigma_u = 0
    sig = np.full((self.__T, self.__T), sigma_u)
    np.fill_diagonal(sig, sigma2)
    matrix = np.kron(np.eye(self.__N), sig)
    return matrix
  
  def build_FE(self) -> pd.DataFrame:
    lr = self.__lr_df.copy(deep=True)
    for i, unit in enumerate(lr.SpUnit.unique(), start=1):
        lr[f'd{i}'] = np.where(lr.SpUnit == unit, 1, 0)
    return lr
  
  def __estimate_lr(self) -> pd.DataFrame:
    if self.__method == 'fix':
      lr_fe = self.build_FE()
      res_lr = sm.OLS(lr_fe.loc[:, 'target'], sm.add_constant(lr_fe.iloc[:, 3:])).fit()
      return res_lr
    else:
      lr_fe = self.build_FE()
      resid = (sm.OLS(lr_fe.loc[:, 'target'], sm.add_constant(lr_fe.iloc[:, 3:])).fit().resid**2) / (self.__N*(self.__T - 1) - self.__exog)
      lr_re_matrix = self.build_GLS(resid)
      return sm.GLS(self.__df.loc[:, 'target'], sm.add_constant(self.__df.loc[:, self.__l]), lr_re_matrix).fit()
  
  def build_sr(self) -> pd.DataFrame:
    pass
    
  
  def __del__(self) -> None:
    pass